<a href="https://colab.research.google.com/github/csabiu/Cosmology_Course/blob/main/practical/Cosmic_Shear.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/csabiu/Cosmology_Course/blob/main/practical/Cosmic_Shear.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cosmic Shear with KiDS DR2
### From galaxy ellipticities to a $\Lambda$CDM prediction

---

Weak gravitational lensing is the most direct probe of the matter distribution in the Universe.  In this practical you will:

1. **Download** a real **KiDS DR2** lensing catalogue from the [Leiden public archive](https://kids.strw.leidenuniv.nl/DR2/lensingcatalogs.php).
2. **Visualise** the data: sky footprint, source redshift distribution $n(z)$, ellipticity histograms, and a teaching "whisker" plot of the spin-2 shear field.
3. **Measure** the two-point shear correlation functions $\xi_\pm(\theta)$ with **TreeCorr**.
4. **Build** a theoretical model: matter power spectrum $P_\delta(k,z)$ from CAMB, the lensing kernel $W_\kappa(\chi)$ &mdash; using the catalogue's own $n(z)$ &mdash; the Limber convergence power spectrum $C_\ell^{\kappa\kappa}$, and the Hankel transforms to $\xi_\pm(\theta)$.
5. **Overplot** theory and data &mdash; in $\Lambda$CDM, the two should agree within shape-noise error bars.

**Prerequisites:** Lectures 10&ndash;12 (gravitational lensing, cosmic shear, the Limber derivation).

**Network requirement:** the KiDS DR2 catalogue is **2.77 GB**.  Plan for 5&ndash;15 minutes of download time on first run.  The file is cached locally so subsequent runs are fast.

**Exercises:** 3 short "try it" prompts + 2 stretch challenges at the end.  Estimated time: 2 hours.

## Part 0 &mdash; Setup

In [ ]:
!pip install -q camb treecorr astropy scipy

In [ ]:
import os, sys, io, time, urllib.request, warnings
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table, vstack
from astropy.cosmology import FlatLambdaCDM
from scipy import integrate, special, interpolate, ndimage
import camb
import treecorr

warnings.filterwarnings('ignore', category=RuntimeWarning)

# Course plot defaults
plt.rcParams.update({
    'font.size': 13,
    'axes.labelsize': 14,
    'axes.titlesize': 15,
    'legend.fontsize': 11,
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# ============================================================
# Planck 2018 fiducial cosmology
# ============================================================
H0      = 67.36
Om0     = 0.3153
Ob0     = 0.0493
ns      = 0.9649
As      = 2.1e-9
sigma8  = 0.811
h       = H0 / 100.0
ckms    = 299792.458

cosmo = FlatLambdaCDM(H0=H0, Om0=Om0, Ob0=Ob0)

# Reproducibility
rng = np.random.default_rng(seed=42)

# Redshift grid (shared by data and theory)
z_grid = np.linspace(0.01, 3.0, 400)

print(f'Planck 2018 fiducial: H0={H0}, Om0={Om0}, sigma8={sigma8}')

## Part 1 &mdash; Theory recap

Three equations carry the whole cosmic-shear pipeline; everything below is one of these three, evaluated.

**Lensing kernel** &mdash; how efficiently a slice of matter at comoving distance $\chi$ lenses sources distributed with $n(\chi)$:
$$\boxed{\;W_\kappa(\chi) = \frac{3\,\Omega_{\rm m}\,H_0^2}{2\,c^2}\,\frac{f_K(\chi)}{a(\chi)}\,\int_\chi^{\chi_H} n(\chi')\,\frac{f_K(\chi'-\chi)}{f_K(\chi')}\,\mathrm d\chi'\;}$$
In a flat universe $f_K(\chi)=\chi$ and $a(\chi)=1/(1+z(\chi))$.

**Limber convergence power spectrum** &mdash; project the 3-D matter power along the line of sight:
$$\boxed{\;C_\ell^{\kappa\kappa} = \int_0^{\chi_H} \frac{\mathrm d\chi}{\chi^2}\,W_\kappa^2(\chi)\,P_\delta\!\bigl(\ell/\chi,\,z(\chi)\bigr)\;}$$

**Hankel transforms** &mdash; the observable real-space 2-point function:
$$\boxed{\;\xi_\pm(\theta) = \frac{1}{2\pi}\int_0^\infty \ell\,\mathrm d\ell\,J_{0,4}(\ell\theta)\,C_\ell^{\kappa\kappa}\;}$$
$\xi_+$ uses $J_0$ (broad sensitivity), $\xi_-$ uses $J_4$ (small-scale weighted).

A more detailed derivation is in **Lecture 11, Section 7** of the course notes.

## Part 2 &mdash; Get the KiDS DR2 shear catalogue

Real data only.  We pull the KiDS DR2 **G15** lensing catalogue (~2.77 GB) from the public Leiden archive.  This is one of three GAMA-region LDAC-FITS files released with Kuijken et al.\ (2015):

| field | URL | size |
|---|---|---|
| G09 | `http://ds.astro.rug.astro-wise.org:8000/KiDS_G09_2015.cat` | 4.8 GB |
| G12 | `http://ds.astro.rug.astro-wise.org:8000/KiDS_G12_2015.cat` | 4.1 GB |
| **G15** | `http://ds.astro.rug.astro-wise.org:8000/KiDS_G15_2015.cat` | **2.77 GB** |

KiDS DR2 LDAC files are *multi-extension* &mdash; alternating `LDAC_IMHEAD` metadata extensions with `LDAC_OBJECTS` data extensions per tile.  The reader below loops through every extension, picks out the object tables, and `vstack`s them.  Column names follow the KiDS schema:
`ALPHA_J2000`, `DELTA_J2000`, `e1`, `e2`, `weight`, `Z_B`.

The catalogue's $n(z)$ &mdash; an empirical photo-$z$ histogram &mdash; is used directly in the theoretical model.  No Smail-type smooth fit.

In [ ]:
KIDS_URL        = 'http://ds.astro.rug.astro-wise.org:8000/KiDS_G15_2015.cat'
KIDS_LOCAL_PATH = '/tmp/kids_dr2_g15.cat'
KIDS_MAX_BYTES  = 3_500_000_000     # 3.5 GB cap
KIDS_MAX_GAL    = 200_000           # subsample for fast TreeCorr (None for all)


def download_with_progress(url, dst, max_bytes, chunk=8 * 1024 * 1024):
    '''Stream a file to disk, printing live progress.'''
    req = urllib.request.Request(url, headers={'User-Agent': 'cosmo-course'})
    with urllib.request.urlopen(req, timeout=60) as resp:
        size = int(resp.headers.get('Content-Length', 0))
        print(f'  Remote size: {size/1e9:.2f} GB')
        if size > max_bytes:
            raise IOError(f'file size {size/1e9:.2f} GB exceeds cap {max_bytes/1e9:.2f} GB')
        with open(dst, 'wb') as f:
            n_done = 0
            while True:
                buf = resp.read(chunk)
                if not buf:
                    break
                f.write(buf)
                n_done += len(buf)
                if size:
                    pct = 100.0 * n_done / size
                    sys.stdout.write(f'\r  Downloaded {n_done/1e9:5.2f} / '
                                     f'{size/1e9:5.2f} GB ({pct:5.1f}%)')
                    sys.stdout.flush()
        print()
    return dst


def _pick_first(colnames, *candidates):
    lookup = {c.lower(): c for c in colnames}
    for cand in candidates:
        if cand.lower() in lookup:
            return lookup[cand.lower()]
    return None


def read_kids_ldac(path):
    '''Read a KiDS LDAC-FITS lensing catalogue. Return a standardised astropy Table.'''
    print(f'  Reading LDAC-FITS at {path} ...')
    tables = []
    with fits.open(path, memmap=True) as hdul:
        for hdu in hdul:
            if hdu.data is None or hdu.name == 'LDAC_IMHEAD':
                continue
            if not hasattr(hdu.data, 'names'):
                continue
            try:
                t = Table(hdu.data)
            except Exception:
                continue
            ra_col  = _pick_first(t.colnames, 'ALPHA_J2000', 'RA', 'RAJ2000')
            dec_col = _pick_first(t.colnames, 'DELTA_J2000', 'DEC', 'DECJ2000')
            e1_col  = _pick_first(t.colnames, 'bias_corrected_e1', 'e1', 'E1')
            e2_col  = _pick_first(t.colnames, 'bias_corrected_e2', 'e2', 'E2')
            w_col   = _pick_first(t.colnames, 'weight', 'WEIGHT')
            z_col   = _pick_first(t.colnames, 'Z_B', 'ZB', 'z_B', 'photoz')
            if None in (ra_col, dec_col, e1_col, e2_col, w_col):
                continue
            sub = Table({
                'ra':     np.asarray(t[ra_col],  dtype=float),
                'dec':    np.asarray(t[dec_col], dtype=float),
                'e1':     np.asarray(t[e1_col],  dtype=float),
                'e2':     np.asarray(t[e2_col],  dtype=float),
                'weight': np.asarray(t[w_col],   dtype=float),
            })
            sub['z'] = np.asarray(t[z_col], dtype=float) if z_col else np.nan
            tables.append(sub)
    if not tables:
        raise IOError('no LDAC_OBJECTS extensions found')
    full = vstack(tables, join_type='exact')
    print(f'  Read {len(full):,} rows from {len(tables)} extensions.')
    return full


def quality_cut_kids(t, max_gal=None):
    '''Apply lensfit + sky-position sanity cuts and optionally subsample.'''
    m = (np.isfinite(t['ra'])  & (t['ra']  > 100) & (t['ra']  < 300.0) &
         np.isfinite(t['dec']) & (t['dec'] > -10) & (t['dec'] < 10)  &
         np.isfinite(t['e1'])  & (np.abs(t['e1']) < 1.0) &
         np.isfinite(t['e2'])  & (np.abs(t['e2']) < 1.0) &
         np.isfinite(t['weight']) & (t['weight'] > 0) & (t['weight'] < 100))
    if 'z' in t.colnames:
        m &= np.isfinite(t['z']) & (t['z'] > 0.1) & (t['z'] < 1.5)
    t2 = t[m]
    print(f'  After quality cuts: {len(t2):,} / {len(t):,} galaxies')
    if max_gal is not None and len(t2) > max_gal:
        ix = rng.choice(len(t2), size=max_gal, replace=False)
        t2 = t2[np.sort(ix)]
        print(f'  Subsampled to {len(t2):,} galaxies for speed.')
    return t2


# ----------------------------------------------------------------
# Load (real KiDS only).
# ----------------------------------------------------------------
print('Loading KiDS DR2 G15 ...')
if not os.path.isfile(KIDS_LOCAL_PATH):
    download_with_progress(KIDS_URL, KIDS_LOCAL_PATH, KIDS_MAX_BYTES)
else:
    sz = os.path.getsize(KIDS_LOCAL_PATH) / 1e9
    print(f'  Found cached file {KIDS_LOCAL_PATH} ({sz:.2f} GB) -- skipping download.')

cat = quality_cut_kids(read_kids_ldac(KIDS_LOCAL_PATH), max_gal=KIDS_MAX_GAL)
print(f'\nLoaded {len(cat):,} galaxies from KiDS DR2 G15')
print(cat[:5])

### 2.1 Empirical source $n(z)$

Bin the catalogue's photo-$z$s onto `z_grid` to get the source distribution used by the theoretical model.  This is *the* $n(z)$ &mdash; no smoothing, no analytic fit.

In [ ]:
def empirical_nz(z_grid, z_data, n_bins=40, z_min=0.05, z_max=1.6):
    '''Histogram-based n(z): no smoothing, just renormalised pdf on z_grid.'''
    hist, edges = np.histogram(z_data, bins=n_bins, range=(z_min, z_max), density=True)
    centres = 0.5 * (edges[:-1] + edges[1:])
    nz = np.interp(z_grid, centres, hist, left=0, right=0)
    nz = np.maximum(nz, 0)
    norm = np.trapz(nz, z_grid)
    if norm > 0:
        nz /= norm
    return nz

nz_norm = empirical_nz(z_grid, cat['z'])
print(f'Mean source redshift <z> = {np.trapz(z_grid * nz_norm, z_grid):.3f}')

## Part 3 &mdash; Visualise the catalogue

Before measuring anything, look at the data.  Bad data look weird in plots before they look weird in numbers.

### 3.1 Sky footprint

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.scatter(cat['ra'], cat['dec'], s=0.5, alpha=0.25, color='C0')
ax.set_xlabel('RA [deg]')
ax.set_ylabel('Dec [deg]')
ax.set_title(f'KiDS DR2 G15 source positions ($N={len(cat):,}$)')
ax.set_xlim(np.percentile(cat['ra'],  0.5), np.percentile(cat['ra'],  99.5))
ax.set_ylim(np.percentile(cat['dec'], 0.5), np.percentile(cat['dec'], 99.5))
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

### 3.2 Empirical redshift distribution

The histogram below is exactly the $n(z)$ used by the lensing kernel in Part 4 &mdash; no smoothing, no analytic fit.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.hist(cat['z'], bins=40, range=(0.05, 1.6), density=True, alpha=0.6, color='C0',
        label=f'KiDS DR2 photo-$z$ ($N={len(cat):,}$)')
ax.plot(z_grid, nz_norm, lw=1.5, color='C3', label='n(z) interpolated to z_grid')
ax.set_xlabel('$z$')
ax.set_ylabel('$n(z)$')
ax.set_xlim(0, 2.5)
ax.legend()
plt.tight_layout()
plt.show()

### 3.3 Ellipticity distribution

The intrinsic ellipticity dispersion is the dominant noise source in cosmic shear.  Typical values are $\sigma_e \approx 0.3$ per component.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, comp, name in zip(axs, ['e1', 'e2'], [r'$e_1$', r'$e_2$']):
    ax.hist(cat[comp], bins=60, color='C0', alpha=0.7)
    ax.axvline(0, color='k', lw=0.8)
    sig = float(np.std(cat[comp]))
    ax.set_xlabel(name)
    ax.set_title(f'{name}, $\\sigma$ = {sig:.3f}')
    ax.set_xlim(-1.2, 1.2)
plt.tight_layout()
plt.show()

print(f'<e1> = {float(np.mean(cat["e1"])): .4f}  <e2> = {float(np.mean(cat["e2"])): .4f}')

### 3.4 The spin-2 shear field &mdash; whisker plot

Lensing distortions form a *spin-2* field: each ellipticity is a headless line.  Zoom into a $3^\circ\!\times\!2.5^\circ$ corner of the G15 field and draw each source as a short segment oriented along $\tfrac12\arctan(e_2/e_1)$.  Coherent patterns are correlated cosmic shear; the noisy hash is shape noise.

In [ ]:
zoom_ra_min, zoom_ra_max = 211.0, 214.0
zoom_dec_min, zoom_dec_max = 0.0, 2.5

in_zoom = ((cat['ra']  >= zoom_ra_min) & (cat['ra']  <= zoom_ra_max) &
           (cat['dec'] >= zoom_dec_min) & (cat['dec'] <= zoom_dec_max))
cat_zoom = cat[in_zoom]

n_show = min(2000, len(cat_zoom))
idx = rng.choice(len(cat_zoom), size=n_show, replace=False)
ra_s  = np.asarray(cat_zoom['ra'])[idx]
dec_s = np.asarray(cat_zoom['dec'])[idx]
e1_s  = np.asarray(cat_zoom['e1'])[idx]
e2_s  = np.asarray(cat_zoom['e2'])[idx]

emod  = np.hypot(e1_s, e2_s)
phi   = 0.5 * np.arctan2(e2_s, e1_s)
ra_span = zoom_ra_max - zoom_ra_min
length = 0.002 * ra_span * (emod / 0.3)
dx = length * np.cos(phi)
dy = length * np.sin(phi)

fig, ax = plt.subplots(figsize=(7.5, 5.5))
for x, y, ddx, ddy in zip(ra_s, dec_s, dx, dy):
    ax.plot([x - ddx, x + ddx], [y - ddy, y + ddy], color='C0', lw=0.8, alpha=0.8)
ax.set_aspect('equal')
ax.set_xlabel('RA [deg]')
ax.set_ylabel('Dec [deg]')
ax.set_xlim(211, 214)
ax.set_ylim(0, 2.5)
ax.set_title(f'Spin-2 shear whiskers (KiDS DR2 G15 zoom, $N={n_show:,}$)')
plt.tight_layout()
plt.show()

## Part 4 &mdash; Theoretical model

Now build the theoretical $\xi_\pm(\theta)$ from first principles, using the empirical $n(z)$ from the data.

### 4.1 Matter power spectrum from CAMB

Call CAMB once and ask for a non-linear matter power $P_\delta(k,z)$ on a $(k,z)$ grid that brackets the integrals we need.

In [ ]:
def get_camb_matter_power(H0=H0, Om0=Om0, Ob0=Ob0, ns=ns, As=As,
                           sigma8_target=sigma8, z_max=3.0, kmax=100.0,
                           n_z=41, nonlinear=True):
    '''Return a CAMB matter-power interpolator P(z, k) renormalised to sigma8_target.'''
    pars = camb.CAMBparams()
    pars.set_cosmology(H0=H0, ombh2=Ob0*(H0/100.0)**2,
                       omch2=(Om0-Ob0)*(H0/100.0)**2)
    pars.InitPower.set_params(ns=ns, As=As)
    pars.set_matter_power(redshifts=np.linspace(0, z_max, n_z), kmax=kmax)
    if nonlinear:
        pars.NonLinear = camb.model.NonLinear_both
    else:
        pars.NonLinear = camb.model.NonLinear_none
    results = camb.get_results(pars)
    sig8_native = results.get_sigma8_0()
    rescale = (sigma8_target / sig8_native)**2
    PK = results.get_matter_power_interpolator(nonlinear=nonlinear,
                                                hubble_units=False, k_hunit=False)
    return PK, rescale, results

PK, rescale, camb_results = get_camb_matter_power()
print(f'CAMB native sigma8 = {camb_results.get_sigma8_0():.4f}; rescaling P(k) by {rescale:.4f}')

fig, ax = plt.subplots(figsize=(7, 4))
k_plot = np.logspace(-3, 2, 400)
for z_p, c in zip([0.0, 0.5, 1.0], ['C0', 'C2', 'C3']):
    Pk = PK.P(z_p, k_plot) * rescale
    ax.loglog(k_plot, Pk, color=c, lw=2, label=f'$z = {z_p}$')
ax.set_xlabel(r'$k$ [Mpc$^{-1}$]')
ax.set_ylabel(r'$P_\delta(k,z)$ [Mpc$^3$]')
ax.legend()
ax.set_title('Non-linear matter power spectrum (CAMB + halofit)')
plt.tight_layout()
plt.show()

### 4.2 The lensing kernel $W_\kappa(\chi)$

Given the empirical source $n(z)$, project to comoving distance and evaluate
$$W_\kappa(\chi) = \frac{3\,\Omega_{\rm m}\,H_0^2}{2\,c^2}\,\chi\,(1+z)\int_\chi^{\chi_H} n(\chi')\,\frac{\chi'-\chi}{\chi'}\,\mathrm d\chi'.$$

In [ ]:
def comoving_arrays(cosmo, z_arr):
    chi  = cosmo.comoving_distance(z_arr).value
    dz_dchi = cosmo.efunc(z_arr) * H0 / ckms
    return chi, dz_dchi

chi_grid, dz_dchi_grid = comoving_arrays(cosmo, z_grid)
chi_max = chi_grid[-1]

nchi = nz_norm * dz_dchi_grid
nchi /= np.trapz(nchi, chi_grid)


def lensing_kernel(chi_eval, chi_grid, nchi, cosmo):
    z_eval = np.interp(chi_eval, chi_grid, z_grid)
    a_eval = 1.0 / (1.0 + z_eval)
    pref = 1.5 * Om0 * (H0/ckms)**2

    W = np.zeros_like(chi_eval)
    for i, chi in enumerate(chi_eval):
        mask = chi_grid > chi
        if not np.any(mask):
            continue
        cg = chi_grid[mask]
        ng = nchi[mask]
        W[i] = np.trapz(ng * (cg - chi) / cg, cg)
    return pref * chi_eval * W / a_eval


chi_eval = np.linspace(5.0, chi_max, 400)
Wk_eval = lensing_kernel(chi_eval, chi_grid, nchi, cosmo)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(chi_eval, Wk_eval, lw=2)
ax.set_xlabel(r'$\chi$ [Mpc]')
ax.set_ylabel(r'$W_\kappa(\chi)$ [Mpc$^{-1}$]')
ax.set_title('Lensing kernel for the empirical KiDS $n(z)$')
plt.tight_layout()
plt.show()

i_peak = int(np.argmax(Wk_eval))
print(f'Kernel peaks at chi = {chi_eval[i_peak]:.0f} Mpc, z = {np.interp(chi_eval[i_peak], chi_grid, z_grid):.2f}')

### 4.3 The Limber convergence power spectrum

Evaluate
$$C_\ell^{\kappa\kappa} = \int_0^{\chi_H} \frac{\mathrm d\chi}{\chi^2}\,W_\kappa^2(\chi)\,P_\delta\!\bigl(\ell/\chi,\,z(\chi)\bigr)$$
on a log-spaced $\ell$ grid.

In [ ]:
def limber_Cl(ell, chi_eval, Wk_eval, PK, rescale, cosmo):
    z_eval = np.interp(chi_eval, chi_grid, z_grid)
    Cl = np.zeros_like(ell, dtype=float)
    for j, l in enumerate(ell):
        if l < 2:
            continue
        k = l / np.maximum(chi_eval, 1e-6)
        Pk = PK.P(z_eval, k, grid=False) * rescale
        Pk = np.where(k > 1e-4, Pk, 0.0)
        integrand = (Wk_eval**2 / np.maximum(chi_eval**2, 1e-6)) * Pk
        Cl[j] = np.trapz(integrand, chi_eval)
    return Cl

ell_arr = np.logspace(0, 4.5, 500)
Cl_arr = limber_Cl(ell_arr, chi_eval, Wk_eval, PK, rescale, cosmo)

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(ell_arr, ell_arr*(ell_arr+1)*Cl_arr/(2*np.pi), lw=2, color='C3')
ax.set_xlabel(r'$\ell$')
ax.set_ylabel(r'$\ell(\ell+1)\,C_\ell^{\kappa\kappa}/(2\pi)$')
ax.set_title('Convergence power spectrum (Limber, $\\Lambda$CDM)')
ax.set_xlim(10, 3e4)
plt.tight_layout()
plt.show()

### 4.4 Hankel transforms to $\xi_\pm(\theta)$

$$\xi_\pm(\theta) = \frac{1}{2\pi}\int_0^\infty \ell\,\mathrm d\ell\,J_{0,4}(\ell\theta)\,C_\ell^{\kappa\kappa}.$$

In [ ]:
def xi_pm_from_Cl(theta_rad, ell, Cl):
    xip = np.zeros_like(theta_rad)
    xim = np.zeros_like(theta_rad)
    for i, th in enumerate(theta_rad):
        x = ell * th
        integrand_p = ell * Cl * special.j0(x)
        integrand_m = ell * Cl * special.jv(4, x)
        xip[i] = np.trapz(integrand_p, ell) / (2*np.pi)
        xim[i] = np.trapz(integrand_m, ell) / (2*np.pi)
    return xip, xim

theta_theory_arcmin = np.logspace(np.log10(0.5), np.log10(300.0), 60)
theta_theory_rad    = np.deg2rad(theta_theory_arcmin / 60.0)
xip_theory, xim_theory = xi_pm_from_Cl(theta_theory_rad, ell_arr, Cl_arr)

print(f'xi_+ at 1 arcmin   = {np.interp(1.0,  theta_theory_arcmin, xip_theory):.3e}')
print(f'xi_+ at 10 arcmin  = {np.interp(10.0, theta_theory_arcmin, xip_theory):.3e}')
print(f'xi_+ at 100 arcmin = {np.interp(100.0, theta_theory_arcmin, xip_theory):.3e}')

## Part 5 &mdash; Measure $\xi_\pm(\theta)$ with TreeCorr

TreeCorr accelerates the $\mathcal O(N^2)$ pair counting with a KD-tree.  For each pair $(i,j)$ at separation $\theta_{ij}$ it rotates the ellipticities into the tangential/cross frame and accumulates
$$\hat\xi_\pm(\theta) = \frac{\sum_{ij} w_i w_j [e_{t,i} e_{t,j} \pm e_{\times,i} e_{\times,j}]}{\sum_{ij} w_i w_j}.$$

In [ ]:
tc_cat = treecorr.Catalog(
    ra=cat['ra'], dec=cat['dec'],
    g1=cat['e1'], g2=cat['e2'],
    w=cat['weight'],
    ra_units='deg', dec_units='deg',
)

gg = treecorr.GGCorrelation(
    min_sep=1.0, max_sep=300.0, nbins=10,
    sep_units='arcmin', bin_slop=0.05,
)
gg.process(tc_cat)

theta_meas = gg.meanr
xip_meas = gg.xip
xim_meas = gg.xim
sigp_meas = np.sqrt(gg.varxip)
sigm_meas = np.sqrt(gg.varxim)

print(f'theta (arcmin):  {theta_meas}')
print(f'xi_+ (10^-5):    {xip_meas * 1e5}')
print(f'xi_- (10^-5):    {xim_meas * 1e5}')

## Part 6 &mdash; Theory vs measurement

Two-panel comparison.  Solid line is the $\Lambda$CDM Limber prediction; points are the TreeCorr measurement with shape-noise error bars.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)

for ax, theta_thy, xi_thy, theta_d, xi_d, sig_d, lab in zip(
        axs,
        [theta_theory_arcmin]*2,
        [xip_theory, xim_theory],
        [theta_meas]*2,
        [xip_meas, xim_meas],
        [sigp_meas, sigm_meas],
        [r'$\xi_+(\theta)$', r'$\xi_-(\theta)$'],
    ):
    ax.errorbar(theta_d, xi_d, yerr=sig_d, fmt='o', color='C0',
                ms=5, capsize=2.5, label='KiDS DR2 G15')
    ax.plot(theta_thy, xi_thy, lw=2, color='C3', label=r'$\Lambda$CDM theory')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$\theta$ [arcmin]')
    ax.set_ylabel(lab)
    #ax.set_ylim(1e-8, 3e-4)
    ax.legend()

axs[0].set_title(r'$\xi_+$ (J$_0$)')
axs[1].set_title(r'$\xi_-$ (J$_4$)')
plt.tight_layout()
plt.show()

xip_th_at_data = np.interp(theta_meas, theta_theory_arcmin, xip_theory)
xim_th_at_data = np.interp(theta_meas, theta_theory_arcmin, xim_theory)
chi2_p = float(np.sum(((xip_meas - xip_th_at_data) / sigp_meas)**2))
chi2_m = float(np.sum(((xim_meas - xim_th_at_data) / sigm_meas)**2))
print(f'Diagonal chi^2(xi_+) = {chi2_p:6.1f}  (N_dof = {len(theta_meas)})')
print(f'Diagonal chi^2(xi_-) = {chi2_m:6.1f}  (N_dof = {len(theta_meas)})')

## Part 7 &mdash; Discussion

A few notes on what we just did:

- The 2-point function is measured on the **real KiDS DR2 G15** footprint with the lensfit ellipticity catalogue.  Agreement with the theory curve is a direct test of $\Lambda$CDM &mdash; the KiDS papers report a $\sim 2\sigma$ low $S_8$ compared to Planck.
- The error bars come from `gg.varxip`, which is the **shape-noise-only** estimate.  A publication-grade analysis would build a covariance from many mock realisations or from a jackknife of the survey footprint.
- The Limber approximation breaks down at very large angles ($\theta\gtrsim 1^\circ$ for the lowest tomographic bin).  KiDS/DES analyses use non-Limber for $\ell \lesssim 30$.

What's missing from this practical, but important in production analyses:

- **Intrinsic alignments** (IA): galaxy shapes are not perfectly randomly oriented.  An NLA-model parameter is marginalised over.
- **Photometric-redshift priors** on the source $n(z)$ &mdash; the largest systematic in Stage-III/IV cosmic shear.
- **Baryonic feedback** suppressing $P_\delta(k)$ at small scales; resolved via scale cuts or HMcode.
- **Multiplicative shear bias** $m$ from shape-measurement calibration.

## Part 8 &mdash; Exercises

### Try it (short)

1.  **Cosmology dependence.**  Rerun Part 4 with $\Omega_{\rm m}=0.25$ (then $\sigma_8 = 0.90$).  How does the theory $\xi_+(\theta)$ shift?  Define $S_8 \equiv \sigma_8\sqrt{\Omega_{\rm m}/0.3}$ and explain why $\xi_+$ is more sensitive to $S_8$ than to either parameter alone.
2.  **Photo-$z$ shift.**  Add a constant offset $\Delta z = \pm 0.02$ to `cat['z']` before computing `nz_norm`.  How does the lensing kernel shift, and how does $\xi_+$ at $\theta=10'$ change?  This is the Stage-III/IV photo-$z$ systematic.
3.  **Linear vs non-linear $P_\delta$.**  Set `nonlinear=False` in `get_camb_matter_power`.  On what angular scales does the difference show up?


## References

- Kuijken et al., MNRAS **454**, 3500 (2015) &mdash; KiDS DR2 lensing pipeline.
- Hildebrandt et al., A&A **633**, A69 (2020) &mdash; KiDS+VIKING-450 cosmology.
- Asgari et al., A&A **645**, A104 (2021) &mdash; KiDS-1000 cosmic shear.
- Bartelmann & Schneider, *Phys. Rep.* **340**, 291 (2001) &mdash; canonical weak-lensing review.
- Jarvis (2015), TreeCorr documentation, [rmjarvis.github.io/TreeCorr](https://rmjarvis.github.io/TreeCorr/).
